# L3 supplement: Pandas and safe joins

Read tables, filter with .loc, clean strings, append compatible tables, and validate joins.

**Environment:** See ../requirements.txt and the labs index. Examples use bundled inputs; no internet is required after dependency installation. Synthetic data is for teaching only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / "datasets").exists() else Path.cwd().parent
assert (ROOT / "datasets").exists(), "Run from the repository root or labs directory; in Colab upload/clone the repository first."
DATA = ROOT / "datasets"


## Selection and string normalization

In [ ]:
sales = pd.read_csv(DATA / "sales_multiyear.csv", parse_dates=["OrderDate"])
print(sales.loc[sales["Quantity"] >= 5, ["OrderID", "Quantity"]].head())
print(pd.Series([" North ", "SOUTH"]).str.strip().str.title())

## Join cardinality and row-count protection

In [ ]:
products = pd.read_csv(DATA / "dim_product.csv")
joined = sales[["OrderID","LineID","ProductKey"]].merge(products, on="ProductKey", how="left", validate="many_to_one", indicator=True)
assert len(joined) == len(sales)
assert joined["_merge"].eq("both").all()
print(joined.head())
appended = pd.concat([sales.iloc[:3],sales.iloc[3:6]],ignore_index=True)
assert len(appended) == 6

## Student task
What would happen if the product dimension contained duplicate ProductKey values?

## Reference solution
Try the task before reading this cell.

In [ ]:
duplicate_dim = pd.concat([products,products.iloc[[0]]])
try:
    sales.merge(duplicate_dim,on="ProductKey",validate="many_to_one")
except pd.errors.MergeError as error:
    print("Expected validation failure:",error)

## Interpretation and limitations
Record the dataset grain, units, assumptions, and a limitation in your submission. Explain one result rather than only submitting screenshots.